# Uniaxial permeation, two-piston: D_c, M and κ from one run

Analysis for `uniaxial_permeation.lmp`: a laterally periodic gel slab resting on a solvent-permeable support,
between a **feed** reservoir and a **permeate** reservoir, each closed by a piston (Marioni et al., J. Membr. Sci. 738
(2026) 124837, Eq. 3; their Fig. 2B rotated by 90°). One run goes through these stages:

| stage | pistons | what it gives | files |
|---|---|---|---|
| reference (Phase 1.5) | both at `P_target` | the zero-flux baseline | `…_ref_…` |
| drive (Phase 2) | feed at `P_target + ΔP`, permeate at `P_target` | **D_c** from the displacement evolution, **M** from the steady displacement profile | `…_<NSTEPS>` |
| velocity-controlled (Phase 2v) | feed moved at the drive's mean velocity, permeate at `P_target` | **κ_v** = q L/ΔP on the live network; ΔP is measured | `…_<NSTEPS>_v` |
| frozen network (Phase 2d) | same feed velocity, polymer stopped | **κ_f** = q L/ΔP on the rigid network | `…_<NSTEPS>_f` |
| unload (Phase 2c, optional) | ΔP ramped back to zero | D_c of the re-swelling | `…_<NSTEPS>_u` |

The run has to reconcile three parameters: the cooperative diffusivity **D_c**, the longitudinal modulus **M** and the
permeability **κ**, which linear poroelasticity ties together as **κ = D_c/M**. D_c and M come from the polymer
displacement alone; κ_v and κ_f come from a flux and a pressure drop alone. Section 2 puts them side by side.

Runs made before 2026-11-16 have no velocity-controlled stage. The notebook then uses the drive's constant-ΔP
permeability in its place, and reads a frozen stage (if there is one) at the applied ΔP.

All code is in `scripts/lib/uniaxial.py`. Notes on the methods are at the end.

## Files (synced by the sync cell)

Into `flow_data_local/permeation/<RUN_ID>/` (cluster `output_files/…`), names `<name>_<DATANAME>_<INTERACTION>_<NSTEPS>`;
the stage files carry the suffix `_v`, `_f` or `_u` after `<NSTEPS>`.

| file | content | stages |
|---|---|---|
| `disp_z_polymer_…` | polymer z-displacement profile since the ramp start (`fix ave/chunk`, binned by current z, every `disp_nfreq` = 50 k steps) → D_c, M | drive, v, u |
| `gel_dimensions_{rg,bb}_…` | gel thickness (Rg-based; bounding box with the extreme beads) | all |
| `permeation_…` | block-averaged `z_feed z_perm F_feed F_perm P_feed P_perm Q_perm N_permeate` | all |
| `permeate_count_…` | solvent beads below the support, every `flux_freq` steps → the flux | all |
| `vctrl_info_…_v` | the prescribed feed-piston velocity and the drive window it was measured over | v |
| `piston_position_…`, `piston_velocity_…`, `piston_force_…`, `piston_force_avg[_ref]_…` | `[feed \| perm]` columns | all |
| `piston_pressure_…` | `P_feed`, `P_perm` measured (`F_fluid/(lx ly)`) and applied (the applied feed value is nominal in v and f) | all |
| `pressure_feed_…`, `pressure_permeate_…` | reservoir virial pressures and densities | all |
| `sigma{zz,xx,yy}_{polymer,solvent}[_ref]_…` | partial stress profiles, kinetic term included (zz only in v and f) | all |
| `solvent_density_z[_ref]_…` | solvent number / mass density profiles | all |
| `support_force_…` | pair force of the polymer on the support | all |
| `strain_zz_…`, `box_dimensions_…`, `polymer_com_…`, `stress_aniso_…` | geometry and diagnostics | drive, u |

Into `flow_data_local/traj_files.nosync/`: `traj_ref_…` (box header, wall planes) and `traj_stress_…` (the drive).

## 1 · Setup and computation

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: uniaxial.py (all analysis code) + volfrac.py
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import uniaxial as uni
uni = importlib.reload(uni)            # pick up edits to lib/uniaxial.py without a kernel restart
uni.setup_style()
print('analysis code: ', LIB / 'uniaxial.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit when switching runs
# ══════════════════════════════════════════════════════════════════════════
cfg = uni.Config(
    # ---- the run ----
    DATANAME    = "final_config_slab_support_periodic_10beads_wide_rho04_random_1.0_1.0_16000000_two_pist_pm40",
    INTERACTION = "1.0_1.0",          # epsSS_epsSP
    NSTEPS      = 4000000,            # drive length = the <steps> tag; None resolves it from the files after the sync.  Pinned here:
                                      # the live drive of this gel will share the DATANAME, and with None the sync takes the newest
                                      # files of any length
    RUN_ID      = "random_10b_wide_perm_fref_1",  # local folder under flow_data_local/{permeation,plots}: job 55745208
                                      # (2026-11-15/16), the wide random 10-bead gel with FROZEN_AT=ref: the network frozen undeformed
                                      # for the whole run, dP 0.1, 4 M steps.  It gives the zero-strain kappa (constant-dP flux) only:
                                      # the displacement is zero by construction (no D_c, no M) and there are no _v, _f or _u files.
                                      # Previous: periodic_rho04_14M_11M_perm_3 (lattice 5-bead gel, NSTEPS None, DATANAME
                                      # final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000002_two_pist_pm40)
    mode        = "permeation",
    DP_PISTON   = None,               # applied dP of the drive; None -> read from piston_pressure
    P_BARO      = 1.5,                # P_target
    # ---- steady windows ----
    plateau_frac      = 0.25,         # trailing fraction of the drive used for the steady profile means
    plateau_frac_auto = 0.45,         # longest candidate window of the drive's flux plateau (drift test)
    # ---- post-drive stages (files _v, _f, _u; each is skipped when the run has none) ----
    PERM_VCTRL = True,                # velocity-controlled stage on the live network -> kappa_v
    PERM_VCTRL_SKIP = 0,              # steps after the switch left out of its window (the drift test on dP picks the window)
    PERM_FROZEN = True,               # frozen network -> kappa_f
    PERM_FROZEN_SKIP = 0,             # steps after the freeze left out of its window
    PERM_UNLOAD = True,               # re-swelling after the ramp-down -> D_c, M of the unload
    PERM_UNLOAD_TRACE_SKIP = 0,       # steps after the ramp-down start left out of the unload thickness-trace fit
    PERM_UNLOAD_L = 'ss',             # the unload fit's L: 'ss' = the compressed steady thickness L_0 (1 + c) | 'ref' = the drive's L_0
    # ---- D_c from the displacement evolution (consolidation fit; see Notes) ----
    PERM_GAP = 1.0,                   # sigma: pinned polymer face z^P = z_support + PERM_GAP
    PERM_L0_MODE = 'face',            # fit L_0 = reference polymer-stress edge - Z^P | 'bb': the bounding box (adds the ~3 sigma the
                                      # face floats above the plate at zero flux)
    PERM_L0_STEPS = 500000,           # steps before the ramp averaged for L_0 (and before the ramp-down for the unload)
    PERM_RIGID = 'contact',           # subtract the network's rigid drop onto the plate, u_0(t) (contact layer) | 'none'
    PERM_PIN_NBINS = 2,               # bins of the contact layer averaged for u_0(t)
    PERM_COORDS = 'lagrangian',       # zeta = (z - u_z - Z^P)/L_0, each bin at its reference position | 'eulerian'
    PERM_FORCING = 'applied',         # dP(t) driving the model: 'applied' (the ramp from piston_pressure) | 'measured' (reservoir
                                      # pressures; needed when the pistons ring after the ramp and the gel is thin enough to follow)
    PERM_DISP_RESET = 'auto',         # where the deck reset displace/atom: 'ramp_start' (decks since 2026-09-28) | 'ramp_end' | 'auto'
    PERM_DC_WINDOW_AVG = True,        # fitted modes averaged over each ave/chunk window (the deck block-averages u_z)
    PERM_DC_N_MODES = 8,              # profile-fit modes (0 -> DC_N_MODES); 5 on a thin gel with few bins
    PERM_DC_BOUNDS = (1e-6, 50.0),    # D_c search interval
    PERM_TRACE_SKIP = 1500000,        # steps after the ramp start left out of the thickness-trace fit (gap closing + piston ringing on
                                      # the full gel; 0 on a thin gel, whose whole transient is over by then)
    DC_N_MODES = 5, DC_TRIM_BINS = 2, DC_FRAC_EARLY = 1.0,   # shared with the compression fit; DC_TRIM_BINS = bins dropped at each end of
                                      # the fit domain (2 = 4 sigma on the full gel, 1 on a thin one)
    # ---- M from the steady displacement profile ----
    PERM_M_PRIMARY = 'lag',           # the M that feeds kappa = D_c/M: 'lag' (per-atom pairing of the steady frames with the zero-flux
                                      # reference; needs traj_ref + traj_stress locally, else falls back to 'prof') | 'prof' (parabola
                                      # fitted to the binned profile) | 'bb' | 'trace' (both carry ~2 sigma of extreme beads)
    M_RECORD = "../../flow_data_local/compression/two_pist_rho04_14M_10M_25M_comp_4_andrelax/M_record.json",  # the compression M the
                                      # check against compression uses; None -> compression/M_record_latest.json.  NOTE: this record is
                                      # the lattice 5-bead gel's; the wide 10-bead gel has no valid compression yet (rerun pending)
    # ---- solvent volume fractions (lib/volfrac.py) and pore sizes (lib/psd.py), on the tessellated drive frames ----
    VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, VOR_EVO_FRAMES = 5,   # ~20 s per frame
    P_CAL = 1.5, P_CAL_MODE = 'pore', # pressure handed to lambda(phi_p, P) per bin: 'pore' (ramp between the measured reservoir
                                      # baselines) | 'const' (P_CAL everywhere) | 'thermo' (local P_th)
    PSD_ENABLE = True, PSD_R_PROBE = 0.5, PSD_GRID = 0.5,   # probe = solvent bead radius; grid 0.5 sigma ~ 7 s per frame
    # ---- figure conventions ----
    PARTIAL_NORM = 'share',           # second figure under the stress figures: back pressure removed, divided by dP | 'raw'
    FLIP_Z      = True,               # z-profiles drawn with the feed on the left (plotting only)
    GEL_SHADE_TO_PISTON = True,       # grey membrane band runs up to the final feed-piston plane
    G_REF       = None,               # shear modulus (shear notebooks) for the dP_th = (4/3)(G/M) dP guide; None = no guide.  The
                                      # lattice gel's was 0.2; for this gel take it from shear_analysis_single (RUN_ID random_10b_wide_shear_1)
    G_COMP_REF  = None,               # compression-mode G for the second guide (lattice gel: 0.05); None until the compression rerun
)

In [ ]:
SYNC, FORCE_SYNC = True, False
if SYNC:
    uni.sync_from_expanse(cfg, force=FORCE_SYNC)

In [ ]:
# R -- the zero-flux reference: geometry, reference profiles, wall planes
# P -- the run: drive (profiles, pistons, flux), the displacement fits and the post-drive stages
R = uni.load_reference(cfg)
P = uni.load_permeation(cfg, R)
uni.add_perm_displacement(cfg, R, P)       # D_c (profile + thickness-trace fits) and M = dP L_ss/(2|u_F|) from the drive
uni.add_perm_vctrl(cfg, R, P)              # velocity-controlled stage: kappa_v = q L/dP_measured, live network (None without _v files)
uni.add_perm_frozen(cfg, R, P)             # frozen network: kappa_f (None without _f files)
uni.add_perm_unload(cfg, R, P)             # re-swelling D_c, M (None without _u files)
uni.add_perm_volume_fractions(cfg, R, P)   # mass fraction / Voronoi / calibrated phi_s: tessellates ~9 frames (~20 s each)
uni.add_perm_psd(cfg, R, P)                # geometric porosity + pore-size distribution on the same frames (~7 s each)
uni.print_perm_summary(cfg, R, P)
uni.perm_reconcile(cfg, R, P);             # D_c, M, the kappas and the closure ratios -> P['recon'] (printed)

## 2 · Results: D_c, M and κ

| parameter | measured from | stage | figure |
|---|---|---|---|
| D_c | evolution of the polymer displacement profile u_z(ζ, t), consolidation fit | drive | 2 |
| M | steady displacement profile, M = ΔP L_ss/(2\|u_F\|) | drive | 3 |
| κ_v | prescribed flux and measured pressure drop, live network | velocity-controlled | 1, 4 |
| κ_f | the same with the network frozen | frozen | 1, 4 |
| D_c/M | the two displacement results combined | drive | 4 |

If the gel is one linear poroelastic material, κ_v = D_c/M. κ_f/κ_v shows what the network's own motion adds to the
steady flow.

In [ ]:
# 1 · The run as a timeline: drive (constant ΔP) → velocity-controlled stage → frozen network.
#     (a) piston displacement; (b) pressure drop measured on the pistons, with each stage's window mean and 95 % interval
#     as a bar (applied value dashed over the drive); (c) solvent flux q_s = (dN_s/dt)/(ρ_s0 A_pist) from sliding and
#     independent windows of cfg.q_win_steps, the prescribed q = |v_feed| as a bar over the stages that have it.
#     Over the drive ΔP is held and q_s carries the thermal noise; over the later stages q_s is held and ΔP carries it.
uni.fig_perm_stages(cfg, R, P);

In [ ]:
# 2 · D_c from the displacement evolution during the drive.  Consolidation between the pinned support and the free feed
#     face: u_z = u_0 at the support z^P, ∂u_z/∂z = 0 at the feed face, u_z = 0 at the start of the ΔP ramp, whose history
#     (piston_pressure) drives the model.  u_0 is the network's rigid drop onto the plate and is subtracted (PERM_RIGID);
#     ζ = (z − u_z − Z^P)/L_0 is the material coordinate (PERM_COORDS).
#     (a) (u_z − u_0)(ζ, t)/L_0 snapshots, (b) data and fitted modes → D_c, (c) the feed-face displacement
#     u_F(t) = L_bb(t) − L_0 with the zero-initial-condition series fit (a second D_c) and the profile fit's u(ζ = 1, t)
uni.fig_perm_Dc(cfg, R, P);

In [ ]:
# 3 · M = ΔP_ext L_ss/(2|u_F|) from the steady displacement profile of the drive (u_ss = −(ΔP L/2M)[ζ² − 2ζ], so
#     u_F = −ΔP L/(2M)).  One point per reading of u_F: 'lag' (per-atom pairing of the steady frames with the zero-flux
#     reference), 'prof' (parabola c ζ(2 − ζ) fitted to the binned profile), 'bb' (bounding-box thickness change) and
#     'trace' (thickness-trace asymptote); the last two carry ~2σ of extreme beads.  Intervals: displacement and measured
#     ΔP_ext in quadrature.  PERM_M_PRIMARY picks the one that feeds κ = D_c/M
uni.fig_perm_M(cfg, R, P);

In [ ]:
# 4 · Reconciliation.  (a) κ: live network at prescribed flux (κ_v), frozen network (κ_f), the drive at constant ΔP
#     (the same quantity as κ_v with the noise in the flux), and D_c/M;  (b) the same statement in D_c: the transient
#     D_c against κ M for each measured κ;  (c) closure ratios, 1 = consistent.  95 % intervals; the profile-fit D_c has
#     no interval of its own, so ratios built on it show the other factors' uncertainty only.  The table is printed by
#     the loading cell (uni.perm_reconcile)
uni.fig_perm_reconcile(cfg, R, P);

## 3 · Checks on the three parameters

In [ ]:
# 5 · The drive's flux at constant ΔP.  (a) piston-velocity trace with the two slope estimates; (b) N_perm(t) with the
#     independent-window slopes that give Q ± SE and the cone of its 95 % interval; (c) D_c from the flux and the φ_s
#     drop, D_c = q_s L/ln[(1 − φ_s^P)/(1 − φ_s^F)], whole gel and per half; (d) q_s(t) = (dN_s/dt)/(ρ_s0 A_pist) from
#     sliding and independent windows, each with its fit interval.  Legends sit under the panels
uni.fig_perm_flux(cfg, R, P);

In [ ]:
# D_c estimates side by side: the transient fits | κ_ΔP·M (flux and the strain from the displacement) | the flux with
# the strain from the φ_s profile (figure 5 c).  Last column: (1 − φ_s⁰) D_c, the value expected with no net solvent flux
uni.perm_Dc_summary(cfg, R, P);

In [ ]:
# 6 · The permeate flux the displacement fits imply, Q(t) = A[(D_c/M) ΔP(t)/L_0 − d⟨u_z⟩/dt], with nothing fitted to the
#     flux: (a) against the measured piston-velocity trace and the N_perm-slope Q (steady level = κ = D_c/M in flux form,
#     transient = the solvent the compaction expels); (b) its time integral against the measured permeate bead count
uni.fig_perm_flux_check(cfg, R, P);

In [ ]:
# 7 · M against the compression deck.  The steady frames are paired per atom with the zero-flux reference, so the rigid
#     drop cancels and the contact layer is included; the measured deformation is compared with what the compression
#     record (cfg.M_RECORD) predicts for the drag load σ'(Z) = ΔP (Z^F − Z)/L_0.  (a) u(Z) measured vs predicted,
#     (b) local strain, (c) the M values.  The verdict (tolerance PERM_M_TOL) is printed and coloured in the title; it
#     also reports the polymer-support pair force / A against ΔP_ext
uni.fig_perm_vs_compression(cfg, R, P);

In [ ]:
# 8 · Unload (runs with UNLOAD=1 only): the free re-swelling after the ΔP ramp-down, same panels as figure 2.  The
#     network stress falls to zero at both faces, so τ_1 = L²/(π² D_c) with L the compressed steady thickness
#     (PERM_UNLOAD_L).  Gives D_c(unload), M(unload) and their ratio; skipped with one line otherwise
uni.fig_perm_Dc_unload(cfg, R, P);

In [ ]:
# TEMPORARY (2026-10-09) -- delete once the D_c discrepancy is resolved.
# Does the barycentric velocity v0(t) bias the permeation D_c?  (a) profile fit with v0 fixed by the support-pinned modes
# vs the strain-PDE fit (v0 projected out), the trace fit and κ_ΔP M;  (b) rigid motion: contact layer u_0(t), data mean
# vs the displacement model;  (c) v0(t) at the support: the measured permeate flux Q/A against the flux the fit implies
uni.fig_perm_v0_check(cfg, R, P);

## 4 · State of the gel during the drive

In [ ]:
# 9 · Wet pistons over the drive: (a) displacement of the feed and permeate pistons; (b) P measured (block-averaged
#     F_fluid/A) vs applied, reservoir virial pressures dotted
uni.fig_perm_pistons(cfg, R, P);

In [ ]:
# 10 · Total stress σ^t_zz, σ^t_xx, σ^t_yy / P_perm: zero-flux reference dashed → drive (cividis) → final (bold).
#      Underneath (PARTIAL_NORM = 'share'): (σ^t − P_perm)/ΔP_ext, which reads 1 in the feed and across the membrane and
#      0 in the permeate for zz
uni.fig_total_stress(cfg, R, P);

In [ ]:
# 11 · Solvent and polymer partial normal stresses with the total.  Underneath (PARTIAL_NORM = 'share'): each species'
#      share of the back pressure removed and divided by ΔP_ext, w_s = σ_s/σ^t per bin and snapshot,
#      σ_s* = (σ_s − w_s P_perm)/ΔP_ext, σ_p* = (σ_p − (1 − w_s) P_perm)/ΔP_ext.  Third figure: σ^t* and σ_s* divided by
#      the calibrated solvent volume fraction
uni.fig_partial_stress(cfg, R, P);

In [ ]:
# 12 · Thermodynamic pressure P_th = −⅓ tr(σ^t) (dotted = P_perm).  Underneath: solvent, polymer and total traces with
#      the back pressure removed, / ΔP_ext.  Dash-dot guides: ΔP_th = (4/3)(G/M) ΔP_ext with G = G_REF and G_COMP_REF,
#      M = this run's
uni.fig_thermo_pressure(cfg, R, P);

In [ ]:
# 13 · Solvent volume fraction φ_s (mass fraction, Voronoi, λ-calibrated Voronoi): (a) zero-flux reference, (b) steady
#      drive (box = in-gel change from the reference), (c) the calibration pressure P_local(z) handed to λ(φ_p, P) and
#      (d) the resulting λ(z).  (c) and (d) need traj_stress locally
uni.fig_perm_volfrac(cfg, R, P);

In [ ]:
# 14 · φ_s evolution over the drive: (a) mass fraction, (b) λ-calibrated Voronoi; reference dashed.  Box: the steady drop
#      across the gel, feed face − permeate face, from a quadratic fit over the interior bins (red line), and the in-gel
#      mean change from the reference
uni.fig_perm_volfrac_evolution(cfg, R, P);

In [ ]:
# 15 · Geometric porosity and pore-size distribution (lib/psd.py, largest included sphere): (a) probe-accessible porosity
#      vs φ_s^cal, (b) mean pore diameter, (c) PSD of the membrane interior (reference vs steady; feed-side and
#      permeate-side halves)
uni.fig_perm_psd(cfg, R, P);

## Notes

### The run and its stages

* **Drive.** `P_feed` is ramped from `P_target` to `P_target + ΔP` in `n_ramp` × `ramp_steps` (100 × 10 k steps; a 5 k-step
  ramp rang the undamped sheets for ~2 M steps) and held for `NSTEPS`. The displacement reference (`displace/atom`) is
  reset at the ramp start and the profile is written through the ramp.
* **Velocity-controlled stage (Phase 2v, 2026-11-16).** After the drive the feed piston is taken off its pressure law and
  moved at a constant velocity: its mean velocity over the last `VCTRL_AVG_STEPS` of the drive (displacement over time;
  40 % of `NSTEPS` by default), or `VFEED` when given. The permeate piston stays at `P_target`; two velocity pistons
  would fix the volume and leave the absolute pressure undefined. The flux is then prescribed and the pressure drop is
  the measured quantity, `κ_v = q L/ΔP_meas`.
* **Why.** A membrane of permeability κ at temperature T has a thermal flux noise `σ_q² = 2 k_BT κ/(A L T_avg)` when the
  pressure drop is held, and a pressure noise `σ_ΔP² = 2 k_BT L/(κ A T_avg)` when the flux is held. Relative to the
  signal the two are the same number, so κ is no more precise in one ensemble than in the other. What changes is where
  the noise sits: at fixed flux `q_s(t)` is flat and the permeate count's slope has only bounded storage terms left
  (feed-reservoir compression, gel breathing). perm_3 at constant ΔP: window scatter of `q_s` 7.9e-5 over 150 k-step
  windows against a floor of 5.1e-5, on a signal of 2.1e-4.
* **No feedback loop.** The velocity is set once and not steered toward a target pressure. A force-driven piston already
  is an integral controller on the pressure error (`dv/dt = A(P_app − P_fluid)/m`); a faster loop puts the noise back
  into the velocity, a slower one returns it as drift. ΔP therefore settles near `dp_piston`, within the precision of
  the drive's flux (5–10 %), and κ uses the measured value. If the prescribed velocity differs from the one the gel was
  at, ΔP relaxes over `τ₁ = L²/(π² D_c)` at the start of the stage; the window's drift test on ΔP leaves that out, and
  `PERM_VCTRL_SKIP` can force it.
* **Frozen network (Phase 2d).** Every polymer bead is stopped (velocities zeroed, forces zeroed, thermostat on the
  solvent alone) and the solvent keeps flowing at the same prescribed feed velocity: `κ_f = q L/ΔP_meas` at a flux
  identical to the live stage's, so `κ_f/κ_v = ΔP_v/ΔP_f`. Runs before 2026-11-16 froze the network at the applied ΔP
  instead (Q from the count's slope). `FROZEN_AT=ref` freezes the reference network before the ramp; that run has no
  D_c, M or later stages.
* **Unload (Phase 2c, optional).** `P_feed` is ramped back to `P_target` and the gel re-swells at zero ΔP. The network
  stress falls to zero at both faces at once, so the strain modes and `τ₁` are those of the drive, with L the compressed
  steady thickness. It gives a second D_c and M (figure 8).
* **Halts.** The deck stops a stage when the feed reservoir thins to `feed_halt_thick` or a piston comes within 2 σ of a
  box face. The feed reservoir has to last through the velocity-controlled and frozen stages.

### D_c from the displacement evolution (figure 2)

* **Model.** Linear 1-D poroelasticity, `∂u_z/∂t = q(t) + D_c ∂²u_z/∂z²`, with `q(t)` the total flux (uniform in z; it
  never enters the fit as a number). The polymer is pinned at the support (`u_z = 0` at `z^P = z_support + PERM_GAP`)
  and the feed face is free (`∂u_z/∂z = 0`). The pistons prescribe the pressure drop, so the network stress at the
  support equals ΔP from the first instant and is zero at the free face: the strain has Dirichlet ends and relaxes with
  `τ₁ = L₀²/(π² D_c)`. Strain modes `sin(kπζ)`, displacement modes `1 − cos(kπζ)`.
* **Forcing.** The applied-ΔP history from `piston_pressure` is a sum of increments `w_j` at `t_j`; each mode responds
  with `H_k(t) = Σ_j w_j [1 − exp(−k²π² D_c (t − t_j)/L₀²)]`, averaged over each `ave/chunk` window because the deck
  block-averages `u_z` (`PERM_DC_WINDOW_AVG`). `PERM_FORCING = 'measured'` uses the reservoir pressures instead, for a
  thin gel that follows the pistons' ringing.
* **Rigid drop and coordinates.** At zero flux the polymer face floats ~3 σ above the plate; the drag closes that gap
  before the first snapshot, so the lowest bins read a constant `u_0 ≈ −3 σ`. `PERM_RIGID = 'contact'` subtracts `u_0(t)`
  (mean of the `PERM_PIN_NBINS` lowest bins) and `PERM_COORDS = 'lagrangian'` places each bin at
  `ζ = (z − u_z − Z^P)/L₀` with `Z^P = z^P − u_0`.
* **Two fits.** (a–b) the snapshots with free modal amplitudes, `u/L₀ = Σ a_k (1 − cos kπζ)[H_k(t) − H_k(t_reset)]`,
  D_c by a log grid followed by Brent; (c) the feed-face trace `u_F(t) = L_bb(t) − L₀` against the exact series
  `u_F(∞) Σ_odd k 8/(kπ)² H_k(t)`, a two-parameter fit with standard errors inflated by the residuals' correlation time.
  `PERM_TRACE_SKIP` leaves the first steps out of the trace fit.
* **Convention.** `tests/dc_convention_test.py` solves the equation by finite differences with each experiment's
  boundary conditions and runs both fitters: profile fit 0.99–1.04×, trace fit 1.01×, compression fit 1.04× the true
  D_c. The compression hold relaxes with `τ₁ = L²/(4π² D_c)` (two drained faces), which is the drainage path, not a
  convention factor.

### M from the steady displacement profile (figure 3)

* The steady state is the parabola `u_ss = −(ΔP L/2M)[ζ² − 2ζ]`, so `u_F = −ΔP L/(2M)` and `M = ΔP L/(2|u_F|)`, with L
  the steady thickness `L_ss` and ΔP the pressure drop measured on the pistons over the drive's steady window.
* Readings of `u_F`: `lag` pairs each atom of the steady frames with the zero-flux reference frame, so the rigid drop
  cancels and the contact layer counts (needs `traj_ref` and `traj_stress` locally); `prof` is the parabola
  `c ζ(2 − ζ)` fitted to the binned steady profile; `bb` and `trace` use the bounding box, whose ends are extreme beads
  and carry ~2 σ of the rigid drop into `|u_F|` (M low by ~20 %).
* Figure 7 compares the measured deformation with what the compression deck's stress–strain record predicts for the
  same drag load.

### κ (figures 1, 4, 5)

* `κ = q L/ΔP` with `L` the Rg-based gel thickness over the stage's window and `q` the volume flux per area. In LJ units
  κ is `σ⁵/(ε τ)`; it is the `κ = k/η` of the compression notebooks.
* **Velocity-controlled and frozen stages.** `q = |v_feed|` (the primary: no flux error) and ΔP = the block-bootstrap
  mean of `P_feed_meas − P_perm_meas` over the longest trailing window in which neither ΔP nor the count's slopes drift
  (up to the whole stage). The flux from the permeate count over the same window is printed as `Q_N/(q A)`; it is 1 up
  to the feed/permeate density ratio when the gel passes what the piston pushes, and departs from 1 when the gel is
  still storing or releasing solvent.
* **Drive, constant ΔP.** `Q = (dN_permeate/dt)/ρ_s,0` from the slope of the permeate count, mean and standard error over
  independent windows of `cfg.q_win_steps` inside the longest drift-free trailing window (at most `plateau_frac_auto`
  of the drive). The block-averaged piston velocity is drawn for context only: its blocks are shorter than the sheet's
  velocity correlation time. ΔP is the measured piston difference over the same window.
* **Pressures.** `P_feed_meas`, `P_perm_meas` are the pair forces of the mobile atoms on each sheet over `l_x l_y`
  (compute group/group). They are valid under both pressure and velocity control. Thermo `press` is meaningless in this
  geometry (vacuum margins in the box). The piston damper acts in the settle and reference window only (`damp_prod=0`).
* **`D_c/M`** uses the profile-fit D_c and the primary M; the trace D_c gives a second value.

### Standing results before the velocity-controlled runs

* perm_3 (full gel, ΔP = 0.1; constant-ΔP drive only): D_c 0.147 (profile) and 0.157 [0.138, 0.176] (trace); M 0.439
  [0.430, 0.449] (`lag`); constant-ΔP κ = 0.231 [0.196, 0.266]; `D_c/M` = 0.335 (profile) and 0.358 (trace), i.e.
  1.45 [1.23, 1.67] times the flux-based κ. The compression sweep's `D_c/M` at ε = 0.10 is 0.134, so the transients
  bracket the flux-based value.
* The quarter-thickness gel at ΔP 0.02–0.05 gave κ 0.30–0.32 against 0.24 at 0.1: κ depends on how far the flow
  compacts the gel.

### Profiles, volume fractions and pore sizes (figures 9–15)

* **Profiles.** z-binning covers the full box. The feed and permeate pore baselines are rebuilt per stress snapshot from
  the measured piston planes, keeping every bin at least `res_wall_margin` (3 σ) clear of a piston sheet (the bin
  touching a piston under-reads σ_zz by ~0.07) and following each snapshot's own gel top.
* **Network stress and osmotic pressure are not drawn.** Terzaghi's split needs the pore pressure inside the membrane,
  which under flow falls from feed to permeate; the deck does not measure the solvent chemical potential yet.
  `fig_network_stress` and `fig_osmotic_pressure` remain in the library as placeholders.
* **Normalised partial stresses (`PARTIAL_NORM = 'share'`).** Per bin, snapshot and component: `w_s = σ_s/σ^t`,
  `σ_s* = (σ_s − w_s P_perm)/ΔP_ext`, `σ_p* = (σ_p − (1 − w_s) P_perm)/ΔP_ext`, `σ^t* = (σ^t − P_perm)/ΔP_ext`, with the
  applied `P_perm` and ΔP of the drive. The signal is 0.1 on a 1.5 baseline, so the normalised curves amplify the raw
  scatter 15×; bins within 3 σ of a piston plane are blanked.
* **ΔP_th guide.** Under uniaxial strain the trace of the transferred load is `(1 − 4G/3M)` of the axial one; the
  dash-dot lines use `G_REF` (shear) and `G_COMP_REF` (compression) with this run's M.
* **Volume fractions.** Mass fraction `φ_s^mf = ρ_s/ρ_s,0`, periodic Voronoi `φ_s^vor` on `traj_stress` frames, and
  `φ_s^cal = min(1, λ φ_s^vor)` with `λ(φ_p, P)` from `scripts/calibration/calibration_lambda.json`. λ needs a local
  pressure per bin: `P_CAL_MODE = 'pore'` hands it the ramp between the measured reservoir baselines. At ΔP = 0.1 the
  choice moves `φ_s^cal` by at most 0.0015.
* **Pore sizes.** Polymer beads only, a grid of spacing `PSD_GRID`; geometric porosity = fraction of grid points whose
  largest empty sphere has radius ≥ `PSD_R_PROBE`; pore diameter = that of the largest sphere containing the point
  (Gelb–Gubbins). The loader prints `ξ = √κ` next to the geometric mean pore diameter.
* **Orientation.** `FLIP_Z = True` draws z-profiles with the feed on the left; it is plotting only.